In [1]:
%env CUDA_VISIBLE_DEVICES=6
import sys
sys.path.append("/home/gaoya/Code_Video/vjepa2-main")

import numpy as np
import torch
from decord import VideoReader


env: CUDA_VISIBLE_DEVICES=6


## VJEPA 2.1

In [2]:
from src.hub.backbones import my_vjepa2_1_vit_large_384
path = "/data/gaoya/ckpt/VJEPA2/vjepa2_1_vitl_dist_vitG_384.pt"

encoder, predictor = my_vjepa2_1_vit_large_384(
    pretrained=True,
    checkpoint_path=path,
    map_location="cpu",
)

device = "cuda"
encoder = encoder.to(device).eval()
predictor = predictor.to(device).eval()

# 2. load official PT preprocessor
processor = torch.hub.load("facebookresearch/vjepa2", "vjepa2_preprocessor")


/data/gaoya/miniconda3/envs/vjepa2/lib/python3.12/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


Loaded encoder from key: ema_encoder
Loaded predictor from key: predictor
<All keys matched successfully>
<All keys matched successfully>


Using cache found in /home/gaoya/.cache/torch/hub/facebookresearch_vjepa2_main


In [3]:

# 3. read video
video_path = "/data/gaoya/AAA_test_video/data__/genesis_sim/train/scene_000000/video/preview.mp4"
vr = VideoReader(video_path)

num_frames = len(vr)
print("num_frames =", num_frames)

# 目标是取最多 64 帧；如果视频不够长，就按真实长度取
target_frames = 64
if num_frames >= target_frames:
    frame_idx = np.linspace(0, num_frames - 1, target_frames, dtype=int)
else:
    frame_idx = np.arange(num_frames)

print("frame_idx =", frame_idx[:10], "...", frame_idx[-10:])

video = vr.get_batch(frame_idx).asnumpy()   # T x H x W x C

with torch.inference_mode():
    video = torch.from_numpy(video).permute(0, 3, 1, 2)   # T x C x H x W
    x = processor(video)[0].to(device).unsqueeze(0)          # 1 x T x C x H x W or expected PT format
    out_patch_features = encoder(x)

print(out_patch_features.shape)

num_frames = 60
frame_idx = [0 1 2 3 4 5 6 7 8 9] ... [50 51 52 53 54 55 56 57 58 59]


/data/gaoya/miniconda3/envs/vjepa2/lib/python3.12/contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)


torch.Size([1, 7680, 1024])


## predictor和encoder做mse

### VJEPA2


In [2]:
from transformers import AutoVideoProcessor, AutoModel
ckpt_path = "/data/gaoya/ckpt/facebook-vjepa2-vitg-fpc64-384/"
model = AutoModel.from_pretrained(ckpt_path, local_files_only=True).to("cuda")
processor = AutoVideoProcessor.from_pretrained(ckpt_path)

Loading weights:   0%|          | 0/843 [00:00<?, ?it/s]

In [9]:
model

VJEPA2Model(
  (encoder): VJEPA2Encoder(
    (embeddings): VJEPA2Embeddings(
      (patch_embeddings): VJEPA2PatchEmbeddings3D(
        (proj): Conv3d(3, 1408, kernel_size=(2, 16, 16), stride=(2, 16, 16))
      )
    )
    (layer): ModuleList(
      (0-39): 40 x VJEPA2Layer(
        (norm1): LayerNorm((1408,), eps=1e-06, elementwise_affine=True)
        (attention): VJEPA2RopeAttention(
          (query): Linear(in_features=1408, out_features=1408, bias=True)
          (key): Linear(in_features=1408, out_features=1408, bias=True)
          (value): Linear(in_features=1408, out_features=1408, bias=True)
          (proj): Linear(in_features=1408, out_features=1408, bias=True)
          (dropout): Dropout(p=0.0, inplace=False)
        )
        (drop_path): Identity()
        (norm2): LayerNorm((1408,), eps=1e-06, elementwise_affine=True)
        (mlp): VJEPA2MLP(
          (fc1): Linear(in_features=1408, out_features=6144, bias=True)
          (activation): GELUActivation()
          (fc

In [3]:
import torch
from torchcodec.decoders import VideoDecoder
import numpy as np


# video_path = "/data/gaoya/AAA_test_video/data__/genesis_sim/train/scene_000000/video/preview.mp4"
video_path = "/home/gaoya/Code_Video/vjepa2-main/assets/holding_phone.mp4"
vr = VideoDecoder(video_path)
# 读取所有视频帧
frame_idx = np.arange(0, min(len(vr), 100))  # choosing some frames. here, you can define more complex sampling strategy
video_frames = vr.get_frames_at(indices=frame_idx).data  # T x C x H x W
# video = processor(video_frames, return_tensors="pt").to(model.device)
# with torch.no_grad():
#     video_embeddings = model.get_vision_features(**video)

# print(video_embeddings.shape)


In [4]:
import torch.nn.functional as F

device = "cuda"


def gather_tokens(x, idx):
    """
    x:   [B, N, D]
    idx: [B, K]
    ->   [B, K, D]
    """
    return torch.gather(x, 1, idx.unsqueeze(-1).expand(-1, -1, x.size(-1)))


def build_future_masks_from_frames(model, inputs, context_frames):
    """
    按“前 context_frames 帧作为上下文，后续帧作为预测目标”构造 token mask。
    这里默认 token 在序列里按时间块连续排列（time-major contiguous）。
    """
    pixel_values = inputs["pixel_values_videos"]   # [B, T, C, H, W]
    print(f"输入视频帧形状: {pixel_values.shape}")
    B, T, C, H, W = pixel_values.shape

    tubelet_size = model.config.tubelet_size
    if T % tubelet_size != 0:
        raise ValueError(
            f"T={T} 不能被 tubelet_size={tubelet_size} 整除，请改成能整除的帧数。"
        )
    if context_frames % tubelet_size != 0:
        raise ValueError(
            f"context_frames={context_frames} 不能被 tubelet_size={tubelet_size} 整除。"
        )
    if not (1 <= context_frames < T):
        raise ValueError(f"context_frames 必须在 [1, {T-1}] 之间。")

    with torch.no_grad():
        gt_all = model.get_vision_features(**inputs)   # [B, N, D]
    print(f"GT视频特征形状: {gt_all.shape}")

    _, N, D = gt_all.shape
    time_steps = T // tubelet_size
    # T = 100
    # tubelet_size = 2
    # time_steps = 50

    if N % time_steps != 0:
        raise RuntimeError(
            f"sequence_length={N} 不能整除 time_steps={time_steps}，"
            "当前 token->时间步映射无法稳定恢复。"
        )

    tokens_per_step = N // time_steps
    # 每个patch有 tokens_per_step（=576） 个token
    context_steps = context_frames // tubelet_size
    # context_frames = 40 
    # context_steps = 20

    # 前 context_steps 个时间块作为 context，后面作为 target
    ctx_token_end = context_steps * tokens_per_step
    print(f"context_steps: {context_steps}, ctx_token_end: {ctx_token_end}")
    # 40个frame作为context，对应20个patch，每个patch有576个token，ctx_token_end=20*576=11520

    context_idx = torch.arange(
        0, ctx_token_end, device=gt_all.device, dtype=torch.long
    ).unsqueeze(0).repeat(B, 1)
    # context token index
    # print(f"context_idx形状: {context_idx.shape}, {context_idx.sum()}")



    target_idx = torch.arange(
        ctx_token_end, N, device=gt_all.device, dtype=torch.long
    ).unsqueeze(0).repeat(B, 1)
    # target token index
    # print(f"target_idx形状: {target_idx.shape}, {target_idx.sum()}")

    gt_future = gather_tokens(gt_all, target_idx)   # [B, N_future, D]
    print(f"gt_future形状: {gt_future.shape}")

    meta = {
        "T": T,
        "tubelet_size": tubelet_size,
        "time_steps": time_steps,
        "tokens_per_step": tokens_per_step,
        "context_steps": context_steps,
        "context_frames": context_frames,
    }
    return context_idx, target_idx, gt_all, gt_future, meta







In [5]:
import torch
import torch.nn.functional as F


def gather_tokens(x, idx):
    """
    x:   [B, N, D]
    idx: [B, K]
    ->   [B, K, D]
    """
    return torch.gather(
        x, dim=1, index=idx.unsqueeze(-1).expand(-1, -1, x.size(-1))
    )


def build_future_masks_from_frames(model, inputs, context_frames):
    """
    用完整视频输入构造：
    - context_idx: 前 context_frames 帧对应的 token 索引
    - target_idx : 后续帧对应的 token 索引
    - gt_future  : 完整视频 encoder 特征中，未来部分的 GT token 特征

    这里的关键实现假设：
    token 序列在时间维上按 block 连续排布（time-major contiguous）。
    """
    pixel_values = inputs["pixel_values_videos"]   # [B, T, C, H, W]
    B, T, C, H, W = pixel_values.shape

    tubelet_size = model.config.tubelet_size
    if T % tubelet_size != 0:
        raise ValueError(
            f"T={T} 不能被 tubelet_size={tubelet_size} 整除，请先裁剪帧数。"
        )
    if context_frames % tubelet_size != 0:
        raise ValueError(
            f"context_frames={context_frames} 不能被 tubelet_size={tubelet_size} 整除。"
        )
    if not (1 <= context_frames < T):
        raise ValueError(f"context_frames 必须在 [1, {T-1}] 之间。")

    # 完整视频的 encoder 特征
    with torch.no_grad():
        gt_all = model.get_vision_features(**inputs)   # [B, N, D]

    _, N, D = gt_all.shape
    time_steps = T // tubelet_size

    if N % time_steps != 0:
        raise RuntimeError(
            f"sequence_length={N} 不能整除 time_steps={time_steps}，"
            "无法稳定恢复 token->时间步映射。"
        )

    tokens_per_step = N // time_steps
    context_steps = context_frames // tubelet_size
    ctx_token_end = context_steps * tokens_per_step

    # 前 context_frames 帧对应的 token
    context_idx = torch.arange(
        0, ctx_token_end, device=gt_all.device, dtype=torch.long
    ).unsqueeze(0).repeat(B, 1)

    # 后续未来帧对应的 token
    target_idx = torch.arange(
        ctx_token_end, N, device=gt_all.device, dtype=torch.long
    ).unsqueeze(0).repeat(B, 1)

    # 完整视频里未来部分的 GT 特征
    gt_future = gather_tokens(gt_all, target_idx)   # [B, N_future, D]

    meta = {
        "T": T,
        "tubelet_size": tubelet_size,
        "time_steps": time_steps,
        "tokens_per_step": tokens_per_step,
        "context_frames": context_frames,
        "context_steps": context_steps,
        "ctx_token_end": ctx_token_end,
        "N": N,
        "D": D,
    }
    return context_idx, target_idx, gt_all, gt_future, meta


def forward_with_masks(model, inputs, context_idx, target_idx):
    """
    针对不同 transformers 版本，兼容几种 mask 传法。
    文档签名里 forward 是 list[Tensor]，但参数说明又写成 Tensor，
    所以这里做兼容尝试。
    """
    trials = [
        # 你原来最常用的形式
        (context_idx, target_idx),
        # 文档参数说明里出现过 [B, patch_size, 1] 形式
        (context_idx.unsqueeze(-1), target_idx.unsqueeze(-1)),
        # 某些版本可能要求 list[Tensor]
        ([context_idx], [target_idx]),
        ([context_idx.unsqueeze(-1)], [target_idx.unsqueeze(-1)]),
    ]

    last_err = None
    for ctx, tgt in trials:
        try:
            with torch.no_grad():
                outputs = model(
                    **inputs,
                    context_mask=ctx,
                    target_mask=tgt,
                    skip_predictor=False,
                )
            return outputs
        except Exception as e:
            last_err = e

    raise RuntimeError(f"所有 mask 传法都失败了，最后一个错误是：\n{last_err}")


@torch.no_grad()
def predict_future_features_with_full_clip(
    model,
    processor,
    video_frames,
    context_frames=40,
):
    """
    现成接口版本：
    - 输入完整视频 video_frames
    - 用前 context_frames 帧作为 predictor 的 context
    - 预测后续帧的 latent
    - 与完整视频抽出的 future GT latent 做比较
    """
    device = next(model.parameters()).device
    inputs = processor(video_frames, return_tensors="pt").to(device)

    context_idx, target_idx, gt_all, gt_future, meta = build_future_masks_from_frames(
        model=model,
        inputs=inputs,
        context_frames=context_frames,
    )

    outputs = forward_with_masks(
        model=model,
        inputs=inputs,
        context_idx=context_idx,
        target_idx=target_idx,
    )

    if outputs.predictor_output is None:
        raise RuntimeError("predictor_output 为空，说明 predictor 没有正确执行。")

    pred_future = outputs.predictor_output.last_hidden_state

    if pred_future.shape != gt_future.shape:
        raise RuntimeError(
            f"predictor 输出形状 {tuple(pred_future.shape)} "
            f"与 GT future 形状 {tuple(gt_future.shape)} 不一致。"
        )

    mse = F.mse_loss(pred_future, gt_future)
    cosine = F.cosine_similarity(pred_future, gt_future, dim=-1).mean()

    return {
        "mse": mse,
        "cosine": cosine,
        "pred_future": pred_future,   # [B, N_future, D]
        "gt_future": gt_future,       # [B, N_future, D]
        "gt_all": gt_all,             # [B, N, D]
        "context_idx": context_idx,   # [B, N_ctx]
        "target_idx": target_idx,     # [B, N_tgt]
        "meta": meta,
    }


# =========================
# 使用示例
# =========================
result = predict_future_features_with_full_clip(
    model=model,
    processor=processor,
    video_frames=video_frames,   # 完整100帧
    context_frames=40,
)

print("MSE:", result["mse"].item())
print("Cosine:", result["cosine"].item())
print("pred_future shape:", result["pred_future"].shape)
print("gt_future shape  :", result["gt_future"].shape)
print("meta             :", result["meta"])
with torch.no_grad():
    feat_full = model.get_vision_features(**processor(video_frames, return_tensors="pt").to(next(model.parameters()).device))
    feat_40   = model.get_vision_features(**processor(video_frames[:40], return_tensors="pt").to(next(model.parameters()).device))

print("feat_full:", feat_full.shape)
print("feat_40  :", feat_40.shape)

MSE: 8.745318412780762
Cosine: 0.5122334957122803
pred_future shape: torch.Size([1, 17280, 1408])
gt_future shape  : torch.Size([1, 17280, 1408])
meta             : {'T': 100, 'tubelet_size': 2, 'time_steps': 50, 'tokens_per_step': 576, 'context_frames': 40, 'context_steps': 20, 'ctx_token_end': 11520, 'N': 28800, 'D': 1408}
feat_full: torch.Size([1, 28800, 1408])
feat_40  : torch.Size([1, 11520, 1408])
